In [1]:
import jax
import jax.numpy as jnp

import grain.python as grain

from pathlib import Path
import tiktoken
from helper import load_stories_from_file

In [2]:
#how to load the stories from tiny stories dataset
file_path = Path("TinyStories-1000.txt")
with open(file_path, "r", encoding="utf-8", errors="replace") as f:
    data = f.read()
    stories = data.split('<|endoftext|>')  # Split stories by '<|endoftext|>'

In [3]:
#show the story
story = stories[0]
print(story)
print(f"total stories: {len(stories) - 1} stories")

One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a button on her shirt.
Lily went to her mom and said, "Mom, I found this needle. Can you share it with me and sew my shirt?" Her mom smiled and said, "Yes, Lily, we can share the needle and fix your shirt."
Together, they shared the needle and sewed the button on Lily's shirt. It was not difficult for them because they were sharing and helping each other. After they finished, Lily thanked her mom for sharing the needle and fixing her shirt. They both felt happy because they had shared and worked together.

total stories: 1000 stories


In [4]:
#initialize the tokenizer
tokenizer = tiktoken.get_encoding("gpt2")

print(f"tokenizer vocab size: {tokenizer.n_vocab}")
print(f"tokenizer special tokens: {tokenizer.special_tokens_set}")

tokenizer vocab size: 50257
tokenizer special tokens: {'<|endoftext|>'}


In [5]:
class StoryDataset:
    def __init__(self,stories, maxlen, tokenizer):
        self.stories = stories
        self.maxlen = maxlen
        self.tokenizer = tokenizer
        self.end_token = tokenizer.encode('<|endoftext|>', \
                        allowed_special={'<|endoftext|>'})[0]
        
    def __len__(self):
        return len(self.stories)
    
    def __getitem__(self, idx):
        story = self.stories[idx]
        tokens = self.tokenizer.encode(story, allowed_special={'<|endoftext|>'})
        if len(tokens) > self.maxlen:
           tokens = tokens[:self.maxlen - 1]
           tokens.append(self.end_token)
            
        tokens.extend([0] * (self.maxlen - len(tokens)))
        return tokens


In [6]:
#shuffle the dataset
shuffled_sampler = grain.IndexSampler(
    num_records=10,
    shuffle=True,
    seed=42,
    shard_options=grain.NoSharding(),    
    num_epochs=1
)

def print_sampler_example(sampler, name):
    print(f"\n{name}")
    for i, idx in enumerate(sampler):
        print(f"Record {i}: {idx}")

print_sampler_example(shuffled_sampler, "Shuffled sampler")


Shuffled sampler
Record 0: RecordMetadata(index=0, record_key=8, rng=Generator(Philox))
Record 1: RecordMetadata(index=1, record_key=6, rng=Generator(Philox))
Record 2: RecordMetadata(index=2, record_key=7, rng=Generator(Philox))
Record 3: RecordMetadata(index=3, record_key=9, rng=Generator(Philox))
Record 4: RecordMetadata(index=4, record_key=0, rng=Generator(Philox))
Record 5: RecordMetadata(index=5, record_key=5, rng=Generator(Philox))
Record 6: RecordMetadata(index=6, record_key=1, rng=Generator(Philox))
Record 7: RecordMetadata(index=7, record_key=2, rng=Generator(Philox))
Record 8: RecordMetadata(index=8, record_key=4, rng=Generator(Philox))
Record 9: RecordMetadata(index=9, record_key=3, rng=Generator(Philox))


In [7]:
# create the dataset in batches 32 stories in 1 batch of total 992 stories 31 batches
batch_op_keep = grain.Batch(
    batch_size=32,
    drop_remainder=False # do not drop the last batch if it has less than 32 stories
)

In [8]:
#create the dataloader

def create_dataloader(
    stories,
    tokenizer,
    maxlen,
    batch_size,
    shuffle = False,
    num_epochs = 1,
    seed = 42,
    worker_count = 0
):
    dataset = StoryDataset(stories, maxlen, tokenizer)
    estimated_batches = len(dataset) // batch_size

    sampler = grain.IndexSampler(
        num_records=len(dataset), # 1,000 stories for this dataset
        shuffle=shuffle,
        seed=seed,
        shard_options=grain.NoSharding(),
        num_epochs=num_epochs
    )
    dataloader = grain.DataLoader(
        data_source=dataset,
        sampler=sampler,
        operations=[
            grain.Batch(batch_size=batch_size, drop_remainder=True)
        ],
        worker_count=worker_count
    )
    
    return dataloader, estimated_batches

In [9]:
stories = load_stories_from_file(
    "TinyStories-10000.txt", 
    max_stories=10000
)

Loading stories from TinyStories-10000.txt...
Loaded 10,000 stories


In [10]:
stories[0]

'One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a button on her shirt.\n\nLily went to her mom and said, "Mom, I found this needle. Can you share it with me and sew my shirt?" Her mom smiled and said, "Yes, Lily, we can share the needle and fix your shirt."\n\nTogether, they shared the needle and sewed the button on Lily\'s shirt. It was not difficult for them because they were sharing and helping each other. After they finished, Lily thanked her mom for sharing the needle and fixing her shirt. They both felt happy because they had shared and worked together.<|endoftext|>'

In [11]:
dataloader, batches_per_epoch = create_dataloader(
    stories=stories,
    tokenizer=tokenizer,
    maxlen=128,
    batch_size=32,
    shuffle=False,
    num_epochs=1,
    seed=42,
    worker_count=0  # Single process for experimentation
)

print(f"\nDataLoader created successfully:")
print(f"Will produce {batches_per_epoch} batches per epoch")


DataLoader created successfully:
Will produce 312 batches per epoch


In [12]:
next(iter(dataloader))

[array([3198, 7454, 3198, 7454, 7454, 7454, 7454, 7454, 7454, 3198, 7454,
        7454, 3198, 7454, 7454, 7454, 7454, 7454, 3198, 7454, 7454, 7454,
        7454, 7454, 7454, 3198, 7454, 7454, 3198, 7454, 7454, 7454]),
 array([ 1110,  2402,  1110,  2402,  2402,  2402,  2402,  2402,  2402,
         1110,  2402,  2402,  1110,  2402,  2402,  2402,  2402,  2402,
         1110,  2402,    11,  2402,  2402,  2402,  2402,  1110,  2402,
         2402, 27737,  2402,  2402,  2402]),
 array([  11,  257,   11,  257,  257,  257,  257,  257,  257,   11,  257,
         257,   11,  257,  257,  257,  257,  257,   11,  257,  612,  257,
         257,  257,  257,   11,  257,  257, 1110,  257,  257,  257]),
 array([257, 640, 257, 640, 640, 640, 640, 640, 640, 257, 640, 640, 257,
        640, 640, 640, 640, 640, 257, 640, 373, 640, 640, 640, 640, 257,
        640, 640,  11, 640, 640, 640]),
 array([ 1310,    11,  1310,    11,    11,    11,    11,    11,    11,
         3049,    11,    11,  2576,    11,    11,